# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
base_url = os.getenv('BASE_URL')

if api_key and api_key.startswith('sk-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")


MODEL = 'gpt-5-nano'

openai = OpenAI(
    base_url=base_url,
    api_key=api_key
)

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/',
 '

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
https://edwar

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'brand/about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'documentation page',
   'url': 'https://huggingface.co/docs/transformers'},
  {'type': 'documentation page',
   'url': 'https://huggingface.co/docs/diffusers'},
  {'type': 'documentation page',
   'url': 'https://huggingface.co/docs/safetensors'},
  {'type': 'endpoints page', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter pag

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 1M+ models
Trending on
this week
Models
moonshotai/Kimi-K2-Thinking
Updated
1 day ago
•
30.3k
•
749
maya-research/maya1
Updated
about 5 hours ago
•
5.26k
•
326
dx8152/Qwen-Edit-2509-Multiple-angles
Updated
3 days ago
•
19.1k
•
392
MiniMaxAI/MiniMax-M2
Updated
2 days ago
•
863k
•
1.22k
deepseek-ai/DeepSeek-OCR
Updated
5 days ago
•
3M
•
2.56k
Browse 1M+ models
Spaces
Running
on
CPU Upgrade
1.78k
1.78k
The Smol Training Playbook: The Secrets to Building World-Class LLMs
📝
Explore loss curves for training LLMs
Running
on
Zero
276
276
Qwen-Image-2509-MultipleAngles
👀
Qwen-Image

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\nmoonshotai/Kimi-K2-Thinking\nUpdated\n1 day ago\n•\n30.3k\n•\n749\nmaya-research/maya1\nUpdated\nabout 5 hours ago\n•\n5.26k\n•\n326\ndx8152/Qwen-Edit-2509-Multiple-angles\nUpdated\n3 days ago\n•\n19.1k\n•\n392\nMiniMaxAI/MiniMax-M2\nUpdated\n2 days ago\n•\n863k\n•\n1.22k\ndeepseek-ai/DeepSeek-OCR\nUpdated\n5 days ago\n•\n3M\n•\n2.56k\nBrowse 1M+ models\nSpaces\nRunning\non\nCPU Upgrade\n1.

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a leading AI community and collaboration platform revolutionizing the future of machine learning (ML). It serves as a vibrant hub where machine learning engineers, data scientists, researchers, and AI enthusiasts come together to create, share, and discover open-source models, datasets, and applications.

With over **1 million models** and **250,000+ datasets**, Hugging Face fosters an open and ethical AI ecosystem that encourages innovation across various data modalities including text, image, video, audio, and even 3D.

---

## Core Offerings

- **Models:** Access and collaborate on an extensive library of state-of-the-art machine learning models. Popular examples include moonshotai/Kimi-K2-Thinking and deepseek-ai/DeepSeek-OCR, updated regularly, with large user bases.
  
- **Datasets:** Discover diverse and frequently updated datasets such as nvidia/PhysicalAI-Autonomous-Vehicles and HuggingFaceFW/finewiki to fuel your AI projects.

- **Spaces:** Host and run ML applications and demos conveniently with community-shared resources that cover a wide range of AI use cases — from image editing to video generation.

- **Enterprise Solutions:** Accelerate your machine learning development with paid compute resources and tailored enterprise offerings designed to scale your AI initiatives efficiently.

---

## Community and Culture

Hugging Face is more than just a platform; it’s a **collaborative movement** dedicated to building a better AI future through openness and inclusivity. The community thrives on sharing knowledge, best practices, and innovations to accelerate machine learning development worldwide.

- **Inclusive and Open:** Emphasis on open source and ethical AI development.
- **Collaborative:** Users can host unlimited public projects, engage in discussions, and build their professional machine learning portfolios.
- **Innovative:** Continuous updates on trending models and datasets encourage creative exploration and adoption of cutting-edge AI technologies.

---

## For Customers and Partners

Hugging Face offers scalable solutions for businesses and research institutions looking to integrate AI into their workflows. Whether you're a startup, an enterprise, or an academic lab, the platform provides:

- Ready-to-use open-source models and datasets to jumpstart AI projects.
- Enterprise-grade support and compute services for production-scale AI deployment.
- Access to a vibrant ecosystem with thousands of contributors and active projects.

---

## Careers and Opportunities

Join Hugging Face and be part of the AI revolution. The company values talent passionate about open source, ethical AI, and community-driven innovation. Working at Hugging Face means:

- Collaborating on world-class open-source AI projects.
- Impacting the global AI community.
- Growing in a dynamic environment that fosters learning and creativity.

Explore career openings directly on their platform — whether you are a machine learning engineer, data scientist, developer, or AI researcher, there's a place for you to make a difference.

---

## Brand Identity

- **Colors:** Sunny Yellow (#FFD21E), Vibrant Orange (#FF9D00), Modern Gray (#6B7280)
- **Logo:** Friendly and approachable, reflecting the community-first spirit.

---

## Get Involved

- Visit [huggingface.co](https://huggingface.co) to explore models, datasets, and applications.
- Sign up to contribute, build, and collaborate with a global AI community.
- Leverage Hugging Face’s open-source tools and enterprise solutions to accelerate your AI projects.

---

Hugging Face — *The Home of Machine Learning* and *The AI Community Building the Future.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is a pioneering AI company and the home of a vibrant machine learning community focused on building the future of artificial intelligence. Serving as a collaborative platform, Hugging Face enables machine learning engineers, scientists, and enthusiasts worldwide to create, discover, and share machine learning models, datasets, and applications. The platform supports a wide array of modalities including text, image, video, audio, and even 3D data.

---

## What We Offer

- **Massive Model Repository:** Explore over 1 million open-source machine learning models available for a variety of AI tasks.
- **Extensive Datasets:** Access and contribute to more than 250,000 datasets to accelerate your machine learning projects.
- **Spaces:** Host and interact with thousands of AI applications powered by community and enterprise solutions.
- **Open Source Stack:** Utilize Hugging Face’s robust open-source tools to speed up your AI development and deployment.
- **Enterprise Solutions:** Tailored paid compute and enterprise-grade offerings to help businesses accelerate their ML initiatives.
- **Community Hub:** A central place to learn, collaborate, and contribute to an ethical and open AI ecosystem.

---

## Company Culture & Community

At Hugging Face, the culture is grounded in openness, collaboration, and ethical AI development. The community is passionate about sharing knowledge and advancing machine learning technology together. The platform fosters inclusivity by empowering both professional researchers and casual ML practitioners to build their portfolios and share their work globally.

- Commitment to open-source principles.
- Focus on ethical and responsible AI.
- Supportive and fast-growing global community.
- Opportunities for continuous learning and collaboration.

---

## Our Customers & Users

Hugging Face serves a diverse range of users including:

- AI researchers and data scientists developing advanced machine learning models.
- Enterprises seeking reliable and scalable AI solutions.
- Developers and engineers building innovative AI-powered applications.
- Educators and learners aiming to deepen their AI expertise.
- Open-source contributors and hobbyists experimenting and sharing AI projects.

Hundreds of thousands of users actively contribute and leverage Hugging Face’s resources, fueling rapid innovation in sectors like autonomous vehicles, natural language processing, computer vision, and more.

---

## Careers at Hugging Face

Join a leading AI community where you can impact the future of machine learning. Hugging Face looks for talented individuals who are passionate about:

- Machine learning research and engineering.
- Open-source software development.
- Building scalable cloud and enterprise AI solutions.
- Community engagement and education.

Working at Hugging Face means collaborating with industry experts in an open, innovative, and mission-driven environment focused on creating ethical AI technologies that benefit society.

---

## Get Started with Hugging Face

- Explore AI applications and the model hub on the platform.
- Start building your machine learning portfolio by sharing models and datasets.
- Engage with a global community pushing the boundaries of AI research and applications.

Visit [huggingface.co](https://huggingface.co) and join the AI community building the future today.

---

### Brand Colors:
- Yellow: #FFD21E
- Orange: #FF9D00
- Gray: #6B7280

---

Hugging Face – Empowering the next generation of machine learning engineers and scientists to build an open and ethical AI future together.

In [21]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the AI community building the future — a leading collaboration platform designed to empower machine learning engineers, scientists, and AI enthusiasts worldwide. It serves as a central hub where users can share, explore, discover, and experiment with over 1 million open-source machine learning models, 250,000+ datasets, and 400,000+ applications spanning text, image, video, audio, and even 3D modalities. Hugging Face is at the heart of the AI revolution, fostering an open and ethical AI future.

---

## What We Offer

### The Platform
- **Models:** Browse, host, and collaborate on thousands of community-contributed machine learning models.
- **Datasets:** Explore large and diverse datasets updated frequently, supporting a variety of use cases including NLP, computer vision, and autonomous vehicles.
- **Spaces:** Create and share interactive ML apps and demos powered by community and enterprise users.
- **Community:** Join a fast-growing community built around open collaboration and innovation in AI and ML.

### Features for Teams and Enterprises
- **Enterprise Hub:** Designed for organizations seeking to scale AI projects with advanced platform capabilities.
- **Security & Controls:** Enterprise-grade security including Single Sign-On (SSO), granular access controls, audit logs, token management, and private storage options.
- **Advanced Compute:** Enhanced compute resources such as ZeroGPU for faster, more scalable model inference and training.
- **Analytics:** Centralized dashboards for usage tracking, billing, and resource management.
- Flexible contracts and dedicated support empower enterprises to innovate securely and efficiently.

---

## Company Culture

Hugging Face nurtures a culture of **collaboration, openness, and ethical AI development.** The platform champions the sharing of knowledge, tools, and datasets in the open-source ecosystem, fostering innovation and learning across the global machine learning community. The company supports a diverse and enthusiastic group of contributors, researchers, engineers, and end users working together to build world-class AI technologies.

---

## Careers & Opportunities

Hugging Face is continuously growing and looking for passionate individuals to join their team. The company offers roles for:
- Machine Learning Engineers and Scientists
- Open Source Developers
- Research Scientists pushing the frontiers of AI
- Enterprise Sales and Customer Success Professionals
- Product and UX Designers focused on community-driven AI tools

Join a vibrant workplace at the cutting edge of AI technology alongside a talented and motivated team.

---

## Why Choose Hugging Face?

- Access to the most comprehensive **open-source ML ecosystem**
- Collaborate with a vibrant global AI community
- Drive innovation with **state-of-the-art tools and compute resources**
- Build professional portfolios in public ML projects
- Enterprise-grade security and scalability for organizational adoption

---

## Join Us

**Explore, collaborate, and accelerate your AI journey with Hugging Face — where the AI community builds the future.**

Visit: [huggingface.co](https://huggingface.co)  
Sign Up for free and start building today.

---

### Brand & Visual Identity

- Signature brand colors: Yellow (#FFD21E, #FF9D00) and Gray (#6B7280)
- Available brand assets including logos in .svg, .png, and .ai formats for consistent branding

---

Hugging Face empowers the next generation of machine learning through open, collaborative, and ethical AI development. Together, we are shaping the future of technology.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>